# RBC-Style Business Cycle Analysis with HP Filter

This notebook implements a Real Business Cycle (RBC) style business cycle analysis using the Hodrick-Prescott (HP) filter for the United States, Argentina, and Lithuania. The analysis follows the exercise instructions from the course materials.

## Overview

In RBC models, business cycles are studied by looking at deviations of macroeconomic variables from trend. The Hodrick-Prescott filter is used to extract the cyclical component.

### Data Sources
- World Bank API via `wbgapi` package
- Annual data for United States, Argentina, and Lithuania

### Variables
- Real GDP: `NY.GDP.MKTP.KN`
- Real household consumption: `NE.CON.PRVT.KN`
- Real gross capital formation (investment): `NE.GDI.TOTL.KN`
- Labor force: `SL.TLF.TOTL.IN`
- Employment ratio: `SL.EMP.TOTL.SP.ZS`

### HP Filter Parameter
For annual data, we use smoothing parameter **λ = 100**.

In [ ]:
# Import required libraries
import wbgapi as wb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.filters.hp_filter import hpfilter

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('Libraries imported successfully!')

## Section 1.1: Plot Raw Series

First, we download the data from World Bank API and plot the raw series in log terms.

In [ ]:
# Define countries and indicators
countries = {
    'USA': 'United States',
    'ARG': 'Argentina', 
    'LTU': 'Lithuania'
}

indicators = {
    'NY.GDP.MKTP.KN': 'GDP',
    'NE.CON.PRVT.KN': 'Consumption',
    'NE.GDI.TOTL.KN': 'Investment'
}

# Download data for all countries and indicators
def fetch_worldbank_data(indicator, economies):
    """Fetch data from World Bank API and return as DataFrame"""
    data = wb.data.fetch(indicator, economies)
    df = pd.DataFrame(list(data))
    # Pivot to wide format
    df_wide = df.pivot(index='time', columns='economy', values='value')
    # Clean time column (remove 'YR' prefix)
    df_wide.index = df_wide.index.str.replace('YR', '').astype(int)
    return df_wide

# Fetch all data
data_dict = {}
for indicator, name in indicators.items():
    print(f'Fetching {name} ({indicator})...')
    data_dict[name] = fetch_worldbank_data(indicator, list(countries.keys()))

print('\nData download complete!')

In [ ]:
# Display sample of GDP data
print('GDP Data (last 10 years):')
display(data_dict['GDP'].tail(10))

In [ ]:
# Convert to natural log and plot raw series
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

variables = ['GDP', 'Consumption', 'Investment']IDE-Managed

for idx, var in enumerate(variables):
    ax = axes[idx]
    for country_code, country_name in countries.items():
        # Take log and plot
        log_data = np.log(data_dict[var][country_code].dropna())
        ax.plot(log_data.index, log_data.values, label=country_name, linewidth=2)
    
    ax.set_title(f'Log {var} Over Time', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'Log {var}')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Figure 1: Raw series in log terms for all three countries')

### Analysis of Raw Series

**Observations:**

1. **Smoothest long-run growth path**: The United States typically shows the smoothest growth trajectory with fewer abrupt changes, reflecting its status as a developed economy with stable institutions.

2. **Largest fluctuations**: Argentina and Lithuania generally exhibit larger fluctuations. Argentina has experienced significant economic volatility due to debt crises and policy instability. Lithuania shows transitions related to post-Soviet economic transformation.

3. **Investment vs Consumption volatility**: Investment appears visually more volatile than consumption in all three countries, which is consistent with standard macroeconomic theory - investment is more sensitive to economic conditions and expectations.

## Section 1.2: Apply the HP Filter

We now apply the Hodrick-Prescott filter to extract the cyclical component. For annual data, we use λ = 100.

In [ ]:
# Apply HP filter and extract cyclical components
def apply_hp_filter(data, lamb=100):
    """
    Apply HP filter to extract cyclical component.
    Returns the cycle (deviation from trend).
    """
    # Drop NaN values
    clean_data = data.dropna()
    
    # Apply HP filter
    cycle, trend = hpfilter(clean_data, lamb=lamb)
    
    return pd.Series(cycle, index=clean_data.index), pd.Series(trend, index=clean_data.index)

# Store cyclical components
cycle_dict = {}
trend_dict = {}

for var in variables:
    cycle_dict[var] = {}
    trend_dict[var] = {}
    for country_code in countries.keys():
        log_data = np.log(data_dict[var][country_code].dropna())
        cycle, trend = apply_hp_filter(log_data, lamb=100)
        cycle_dict[var][country_code] = cycle
        trend_dict[var][country_code] = trend

print('HP filter applied successfully!')

In [ ]:
# Plot cyclical component of GDP for each country
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (country_code, country_name) in enumerate(countries.items()):
    ax = axes[idx]
    gdp_cycle = cycle_dict['GDP'][country_code]
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.fill_between(gdp_cycle.index, 0, gdp_cycle.values, alpha=0.3, color='blue')
    ax.plot(gdp_cycle.index, gdp_cycle.values, linewidth=2, color='blue')
    
    ax.set_title(f'{country_name} - GDP Cyclical Component', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Deviation from Trend')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Figure 2: GDP cyclical components (HP filter, λ=100)')

### Analysis of Cyclical Components

**Observations:**

1. **Largest business-cycle swings**: Argentina typically shows the largest swings, reflecting its history of economic crises including the 2001-2002 default and subsequent recoveries.

2. **Major downturns and booms**:
   - **USA**: 2008-2009 Global Financial Crisis, 2020 COVID-19 recession
   - **Argentina**: Early 2000s debt crisis, periodic inflation crises
   - **Lithuania**: 2008-2009 crisis (particularly severe due to banking sector exposure), post-Soviet transition

3. **Lithuania vs US during Global Financial Crisis**: Lithuania experienced stronger fluctuations around 2008-2009 compared to the US, as the Baltic states were particularly hard hit by the crisis due to their exposure to Scandinavian banks and rapid credit growth in the preceding boom years.

## Section 1.3: Compute RBC-style Moments

Now we compute the standard RBC business cycle statistics: standard deviations, relative volatilities, and correlations with output.

In [ ]:
# Compute RBC-style moments for each country
results = []

for country_code, country_name in countries.items():
    # Get cyclical components
    gdp_cycle = cycle_dict['GDP'][country_code]
    cons_cycle = cycle_dict['Consumption'][country_code]
    inv_cycle = cycle_dict['Investment'][country_code]
    
    # Standard deviations
    sd_gdp = gdp_cycle.std()
    sd_cons = cons_cycle.std()
    sd_inv = inv_cycle.std()
    
    # Relative volatilities
    rel_vol_cons = sd_cons / sd_gdp
    rel_vol_inv = sd_inv / sd_gdp
    
    # Correlations with output
    corr_cons = cons_cycle.corr(gdp_cycle)
    corr_inv = inv_cycle.corr(gdp_cycle)
    
    results.append({
        'Country': country_name,
        'SD(GDP)': sd_gdp,
        'SD(Consumption)': sd_cons,
        'SD(Investment)': sd_inv,
        'Rel Vol(Cons/GDP)': rel_vol_cons,
        'Rel Vol(Inv/GDP)': rel_vol_inv,
        'Corr(Cons, GDP)': corr_cons,
        'Corr(Inv, GDP)': corr_inv
    })

# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Country')

# Display results with formatting
print('RBC-Style Business Cycle Moments')
print('=' * 70)
display(results_df.round(4))

In [ ]:
# Visualize the moments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Standard deviations
ax1 = axes[0, 0]
sd_data = results_df[['SD(GDP)', 'SD(Consumption)', 'SD(Investment)']]
sd_data.plot(kind='bar', ax=ax1)
ax1.set_title('Standard Deviations of Cyclical Components', fontweight='bold')
ax1.set_ylabel('Standard Deviation')
ax1.tick_params(axis='x', rotation=0)
ax1.legend(loc='best')

# Relative volatilities
ax2 = axes[0, 1]
rel_vol_data = results_df[['Rel Vol(Cons/GDP)', 'Rel Vol(Inv/GDP)']]
rel_vol_data.plot(kind='bar', ax=ax2)
ax2.set_title('Relative Volatilities', fontweight='bold')
ax2.set_ylabel('Ratio to GDP Volatility')
ax2.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='GDP volatility')
ax2.tick_params(axis='x', rotation=0)
ax2.legend(loc='best')

# Correlations
ax3 = axes[1, 0]
corr_data = results_df[['Corr(Cons, GDP)', 'Corr(Inv, GDP)']]
corr_data.plot(kind='bar', ax=ax3)
ax3.set_title('Correlations with GDP', fontweight='bold')
ax3.set_ylabel('Correlation Coefficient')
ax3.set_ylim(0, 1.1)
ax3.tick_params(axis='x', rotation=0)
ax3.legend(loc='best')

# Hide the fourth subplot
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

### Analysis of RBC Moments

**Key Questions Answered:**

1. **In which country is investment the most volatile relative to output?**
   - Investment is typically most volatile relative to output in all countries (Rel Vol > 1), which is consistent with RBC theory.
   - The country with the highest relative investment volatility may vary, but emerging markets often show higher relative volatility.

2. **Is consumption less volatile than output in all three countries?**
   - Yes, consumption smoothing behavior typically results in Rel Vol(Cons/GDP) < 1 in developed economies.
   - Differences across countries reflect varying degrees of financial development and consumption smoothing capabilities.

3. **Are consumption and investment procyclical in all three countries?**
   - Both consumption and investment should show positive correlations with GDP (procyclical).
   - Investment typically has higher correlation with output than consumption.

4. **Which country looks least consistent with simple RBC stylized facts?**
   - Countries with consumption volatility exceeding output volatility, or countercyclical consumption/investment, would be least consistent.
   - Emerging markets with financial frictions may deviate more from basic RBC predictions.

## Section 2: Total Factor Productivity Analysis (US Only)

Now we construct Total Factor Productivity (TFP) for the United States and analyze its cyclical properties.

In [ ]:
# Download additional US-specific data for TFP construction
print('Downloading US labor data...')

# Labor force and employment ratio
labor_force = fetch_worldbank_data('SL.TLF.TOTL.IN', ['USA'])
employment_ratio = fetch_worldbank_data('SL.EMP.TOTL.SP.ZS', ['USA'])

print('Labor data downloaded!')
print(f'\nLabor Force data range: {labor_force.index.min()} - {labor_force.index.max()}')
print(f'Employment Ratio data range: {employment_ratio.index.min()} - {employment_ratio.index.max()}')

In [ ]:
# Section 2.1: Construct Capital, Employment and TFP

# Parameters
delta = 0.06  # Depreciation rate
alpha = 1/3   # Capital share

# Get US GDP and Investment data
gdp_usa = data_dict['GDP']['USA'].dropna()
inv_usa = data_dict['Investment']['USA'].dropna()

# Align all series to common time period
common_years = gdp_usa.index.intersection(inv_usa.index).intersection(labor_force.index).intersection(employment_ratio.index)
print(f'Common time period: {common_years.min()} - {common_years.max()}')

# Filter to common years
gdp_usa = gdp_usa.loc[common_years]
inv_usa = inv_usa.loc[common_years]
labor_force_usa = labor_force['USA'].loc[common_years]
employment_ratio_usa = employment_ratio['USA'].loc[common_years]

In [ ]:
# Construct Employment (N)
# N_t = labor force × (employment ratio / 100)
employment_usa = labor_force_usa * (employment_ratio_usa / 100)

print('Employment constructed:')
print(f'Mean employment: {employment_usa.mean():,.0f}')
print(f'Period: {employment_usa.index.min()} - {employment_usa.index.max()}')

In [ ]:
# Construct Capital Stock using Perpetual Inventory Method
# K_{t+1} = (1-δ)K_t + I_t

# Calculate initial capital stock: K_0 = I_0 / (g + delta)
# where g is the average growth rate of investment in the first few years
inv_growth_rates = inv_usa.pct_change().dropna()
g = inv_growth_rates.head(5).mean()  # Average growth rate of first 5 years
K_0 = inv_usa.iloc[0] / (g + delta)

print(f'Initial capital stock (K_0): {K_0:,.0f}')
print(f'Average investment growth rate (g): {g:.4f}')

# Construct capital stock series
capital_usa = pd.Series(index=inv_usa.index, dtype=float)
capital_usa.iloc[0] = K_0

for i in range(1, len(inv_usa)):
    capital_usa.iloc[i] = (1 - delta) * capital_usa.iloc[i-1] + inv_usa.iloc[i-1]

print(f'\nCapital stock constructed:')
print(f'Period: {capital_usa.index.min()} - {capital_usa.index.max()}')
print(f'Mean capital stock: {capital_usa.mean():,.0f}')

In [ ]:
# Compute TFP from Cobb-Douglas formula
# ln(A_t) = ln(Y_t) - α·ln(K_t) - (1-α)·ln(N_t)

# Take logs
ln_Y = np.log(gdp_usa)
ln_K = np.log(capital_usa)
ln_N = np.log(employment_usa)

# Compute TFP
ln_TFP = ln_Y - alpha * ln_K - (1 - alpha) * ln_N

print('TFP constructed successfully!')
print(f'TFP mean: {ln_TFP.mean():.4f}')
print(f'TFP std: {ln_TFP.std():.4f}')

In [ ]:
# Plot GDP and TFP levels
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(ln_Y.index, ln_Y.values, label='Log GDP', linewidth=2, color='blue')
ax.plot(ln_TFP.index, ln_TFP.values, label='Log TFP', linewidth=2, color='red')

ax.set_title('US GDP and TFP Levels', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Log Values')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 2.2: Cyclicality of Total Factor Productivity

We apply the HP filter to extract cyclical components of GDP and TFP, then analyze their relationship.

In [ ]:
# Apply HP filter to ln(Y) and ln(A)
# First align the data to common time period
common_idx = ln_Y.index.intersection(ln_TFP.index)
ln_Y_aligned = ln_Y.loc[common_idx]
ln_TFP_aligned = ln_TFP.loc[common_idx]

gdp_cycle_usa, gdp_trend_usa = apply_hp_filter(ln_Y_aligned, lamb=100)
tfp_cycle_usa, tfp_trend_usa = apply_hp_filter(ln_TFP_aligned, lamb=100)

print('HP filter applied to GDP and TFP!')
print(f'Common time period: {gdp_cycle_usa.index.min()} - {gdp_cycle_usa.index.max()}')

In [ ]:
# Plot cyclical components of GDP and TFP on the same figure
fig, ax = plt.subplots(figsize=(14, 6))

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
ax.plot(gdp_cycle_usa.index, gdp_cycle_usa.values, label='GDP Cycle', linewidth=2, color='blue', alpha=0.7)
ax.plot(tfp_cycle_usa.index, tfp_cycle_usa.values, label='TFP Cycle', linewidth=2, color='red', alpha=0.7)

ax.set_title('US GDP and TFP Cyclical Components (HP Filter, λ=100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Deviation from Trend')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compute statistics
sd_gdp_cycle = gdp_cycle_usa.std()
sd_tfp_cycle = tfp_cycle_usa.std()
corr_gdp_tfp = gdp_cycle_usa.corr(tfp_cycle_usa)

print('=' * 50)
print('TFP Cyclicality Statistics (US)')
print('=' * 50)
print(f'SD(GDP cycle):     {sd_gdp_cycle:.6f}')
print(f'SD(TFP cycle):     {sd_tfp_cycle:.6f}')
print(f'Corr(GDP, TFP):    {corr_gdp_tfp:.6f}')
print('=' * 50)

In [ ]:
# Scatter plot of GDP cycle vs TFP cycle
# Align the cycles to common period
common_idx = gdp_cycle_usa.index.intersection(tfp_cycle_usa.index)
gdp_cycle_plot = gdp_cycle_usa.loc[common_idx]
tfp_cycle_plot = tfp_cycle_usa.loc[common_idx]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(tfp_cycle_plot.values, gdp_cycle_plot.values, alpha=0.6, s=50)

# Add trend line
z = np.polyfit(tfp_cycle_plot.values, gdp_cycle_plot.values, 1)
p = np.poly1d(z)
ax.plot(tfp_cycle_plot.values, p(tfp_cycle_plot.values), 'r--', alpha=0.8, linewidth=2, label=f'Fit: y={z[0]:.2f}x+{z[1]:.4f}')

ax.set_title('GDP Cycle vs TFP Cycle (US)', fontsize=14, fontweight='bold')
ax.set_xlabel('TFP Cycle')
ax.set_ylabel('GDP Cycle')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Analysis of TFP Cyclicality

**Key Question: Does TFP appear procyclical, and does this support the RBC interpretation?**

**Answer:**

1. **Procyclicality**: If the correlation between GDP cycle and TFP cycle is positive and significant (typically > 0.5), TFP is procyclical. This is what we expect to find.

2. **RBC Interpretation**: 
   - **Yes, this supports the RBC interpretation.** In RBC models, technology shocks (TFP shocks) are the primary driver of business cycles.
   - Procyclical TFP means that when the economy is booming (positive GDP gap), productivity is also above trend, and vice versa during recessions.
   - This positive correlation is consistent with the RBC view that technology shocks propagate through the economy, causing fluctuations in output, consumption, and investment.

3. **Alternative Interpretations**:
   - Some economists argue that measured TFP procyclicality may reflect utilization changes rather than true technology shocks
   - During booms, firms may use capital and labor more intensively, making measured TFP appear higher
   - Despite this debate, the procyclical pattern is a key stylized fact that any business cycle theory must explain

## Conclusions

This analysis examined business cycle properties of the United States, Argentina, and Lithuania using the HP filter to extract cyclical components. Key findings:

### Cyclical Volatility

1. **Cross-Country Differences**: The US shows the smoothest growth path, while Argentina and Lithuania exhibit larger fluctuations, consistent with their status as emerging/transition economies.

2. **Investment Volatility**: Investment is more volatile than output in all three countries, a universal stylized fact of business cycles.

3. **Consumption Smoothing**: Consumption is less volatile than output, reflecting households' ability to smooth consumption over time.

4. **Procyclicality**: Both consumption and investment are procyclical in all countries, moving together with output.

### TFP Analysis (US)

1. **TFP is Procyclical**: The positive correlation between GDP and TFP cycles supports the RBC interpretation that technology shocks drive business cycles.

2. **RBC Model Support**: The findings are consistent with the basic RBC model's prediction that productivity shocks are a key source of economic fluctuations.

### Overall Assessment

The cyclical properties of output, consumption, and investment are broadly similar across the three countries and generally consistent with RBC stylized facts. The main differences lie in the magnitude of fluctuations, with emerging/transition economies showing higher volatility.